# CDC de Tramitação — SCD Type 2

## Objetivo
Rastrear a evolução de status de proposições legislativas usando
Change Data Capture com SCD Type 2, permitindo reconstrução histórica
via Delta Time Travel.

## Conceitos
- **SCD Type 2:** cada mudança gera uma nova linha, preservando o histórico
- **valid_from / valid_to:** período em que um status esteve ativo
- **is_current:** indica o status atual (último registro)
- **Time Travel:** recurso do Delta Lake para consultar dados em qualquer data

In [0]:
import requests
import json

# Buscar uma proposicao com tramitacao rica
# Usamos a API de proposicoes com palavras-chave
url = "https://dadosabertos.camara.leg.br/api/v2/proposicoes"
params = {
    "itens": 5,
    "dataInicio": "2023-01-01",
    "dataFim": "2023-03-31",
    "keywords": "reforma tributária",
    "ordem": "ASC",
    "ordenarPor": "id"
}

response = requests.get(url, params=params)
print(f"Status: {response.status_code}")

if response.status_code == 200:
    dados = response.json()
    print(f"Proposicoes encontradas: {len(dados['dados'])}")
    if dados['dados']:
        for p in dados['dados']:
            print(f"  ID: {p['id']} - {p['siglaTipo']} {p['numero']}/{p['ano']}")
            print(f"  Ementa: {p['ementa'][:150]}")
            print(f"  Data apresentacao: {p['dataApresentacao']}")
            print()
else:
    print(f"Erro: {response.text}")

In [0]:
import requests
import json

id_prop = 2258196
url = f"https://dadosabertos.camara.leg.br/api/v2/proposicoes/{id_prop}/tramitacoes"

response = requests.get(url)
print(f"Status: {response.status_code}")

if response.status_code == 200:
    dados = response.json()
    # Se for uma lista direta, pegar os primeiros 5
    if isinstance(dados, list):
        tramitacoes = dados[:5]
    else:
        tramitacoes = dados.get('dados', [])[:5]
    
    print(f"Tramitacoes retornadas: {len(tramitacoes)}")
    for t in tramitacoes:
        print(f"\nData/Hora: {t.get('dataHora')}")
        print(f"Descricao: {t.get('descricaoTramitacao', t.get('descricao', ''))[:200]}")
        print(f"Despacho: {t.get('despacho', '')[:200]}")
        print(f"Orgao: {t.get('siglaOrgao', 'N/A')}")
        print(f"Regime: {t.get('regime', 'N/A')}")
        print(f"Sequencia: {t.get('sequencia', 'N/A')}")
else:
    print(f"Erro: {response.text}")

# Coleta de Tramitações de Proposições
Busca tramitações de proposições selecionadas para implementar SCD Type 2.
Amostra de 10 proposições com tramitação rica para demonstração do conceito.

In [0]:
import requests
import time

# Proposicoes que encontramos + algumas relevantes
ids_proposicoes = [
    2258196,  # PL 3887/2020 - Reforma Tributária (CBS)
    2320038,  # PL 952/2022
    2196833,  # PEC 45/2019 - Reforma Tributária
]

# Tambem podemos pegar algumas proposicoes de destaque de 2023
# Buscar proposicoes do periodo das CPIs que ja conhecemos
url_busca = "https://dadosabertos.camara.leg.br/api/v2/proposicoes"
params_busca = {
    "itens": 10,
    "dataInicio": "2023-05-01",
    "dataFim": "2023-06-30",
    "keywords": "lei",
    "ordem": "ASC",
    "ordenarPor": "id"
}

response_busca = requests.get(url_busca, params=params_busca)
if response_busca.status_code == 200:
    dados_busca = response_busca.json()
    for p in dados_busca.get('dados', []):
        ids_proposicoes.append(p['id'])

# Remover duplicatas
ids_proposicoes = list(set(ids_proposicoes))
print(f"Total de proposicoes a coletar: {len(ids_proposicoes)}")

# Coletar tramitacoes
todas_tramitacoes = []
erros = []
inicio = time.time()

for i, id_prop in enumerate(ids_proposicoes, start=1):
    url = f"https://dadosabertos.camara.leg.br/api/v2/proposicoes/{id_prop}/tramitacoes"
    response = requests.get(url)
    
    if response.status_code == 200:
        dados = response.json()
        if isinstance(dados, list):
            tramitacoes = dados
        else:
            tramitacoes = dados.get('dados', [])
        
        for t in tramitacoes:
            todas_tramitacoes.append((
                id_prop,
                t.get('dataHora'),
                t.get('descricaoTramitacao', ''),
                t.get('despacho', ''),
                t.get('siglaOrgao', ''),
                t.get('regime', ''),
                int(t.get('sequencia', 0)) if t.get('sequencia') else 0
            ))
    else:
        erros.append(id_prop)
    
    if i % 3 == 0 or i == 1 or i == len(ids_proposicoes):
        decorrido = time.time() - inicio
        print(f"  {i}/{len(ids_proposicoes)} | {len(todas_tramitacoes)} tramitacoes | {decorrido:.0f}s")

print(f"\nConcluido! {len(todas_tramitacoes)} tramitacoes, {len(erros)} erros")

# SCD Type 2 — Construção da Tabela de Tramitação

## Lógica
1. Ordenar tramitações por proposição e data
2. Para cada mudança de status, fechar o registro anterior (valid_to = data da mudança)
3. Criar novo registro com o novo status (valid_from = data da mudança, valid_to = 9999-12-31)
4. is_current = true apenas para o registro mais recente

## Exemplo
| valid_from | valid_to | status | is_current |
|------------|----------|--------|------------|
| 2020-07-21 | 2020-07-30 | Apresentação | false |
| 2020-07-30 | 9999-12-31 | Abertura de Prazo | true |

In [0]:
from pyspark.sql.types import StructType, StructField, LongType, StringType, IntegerType
from datetime import datetime

data_coleta = datetime.now().isoformat()

# Preparar dados brutos
dados_tram = []
for t in todas_tramitacoes:
    dados_tram.append((
        t[0], t[1], t[2], t[3], t[4], t[5], t[6], data_coleta
    ))

schema = StructType([
    StructField("id_proposicao", LongType(), True),
    StructField("data_hora", StringType(), True),
    StructField("descricao_tramitacao", StringType(), True),
    StructField("despacho", StringType(), True),
    StructField("sigla_orgao", StringType(), True),
    StructField("regime", StringType(), True),
    StructField("sequencia", IntegerType(), True),
    StructField("data_coleta", StringType(), True)
])

df_tram = spark.createDataFrame(dados_tram, schema=schema)
spark.sql("DROP TABLE IF EXISTS bronze.tramitacoes")
df_tram.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze.tramitacoes")

print(f"Bronze: {df_tram.count()} registros salvos.")

In [0]:
%sql
CREATE OR REPLACE TABLE prata.tramitacoes_scd2
USING DELTA
COMMENT 'Camada Prata - SCD Type 2 de tramitacoes de proposicoes'
AS
WITH tramitacoes_ordenadas AS (
    SELECT 
        id_proposicao,
        CAST(data_hora AS TIMESTAMP) AS data_hora,
        descricao_tramitacao,
        despacho,
        sigla_orgao,
        regime,
        sequencia,
        ROW_NUMBER() OVER (PARTITION BY id_proposicao ORDER BY CAST(data_hora AS TIMESTAMP), sequencia) AS rn
    FROM bronze.tramitacoes
),
com_lead AS (
    SELECT 
        *,
        LEAD(data_hora) OVER (PARTITION BY id_proposicao ORDER BY data_hora, sequencia) AS proxima_data
    FROM tramitacoes_ordenadas
)
SELECT 
    id_proposicao,
    data_hora AS valid_from,
    COALESCE(proxima_data, CAST('9999-12-31' AS TIMESTAMP)) AS valid_to,
    CASE WHEN proxima_data IS NULL THEN TRUE ELSE FALSE END AS is_current,
    descricao_tramitacao AS status,
    despacho,
    sigla_orgao,
    regime,
    sequencia
FROM com_lead
ORDER BY id_proposicao, valid_from

In [0]:
%sql
-- Verificar SCD Type 2 de uma proposicao especifica
SELECT 
    id_proposicao,
    valid_from,
    valid_to,
    is_current,
    status,
    sigla_orgao,
    sequencia
FROM prata.tramitacoes_scd2
WHERE id_proposicao = 2258196
ORDER BY valid_from

# Time Travel — Reconstrução Histórica
Consultar o estado de uma proposição em qualquer data usando Delta Time Travel.

In [0]:
%sql
-- Estado da PL 3887 em 15/09/2020
SELECT 
    id_proposicao,
    valid_from,
    valid_to,
    status,
    despacho,
    sigla_orgao
FROM prata.tramitacoes_scd2
WHERE id_proposicao = 2258196
  AND valid_from <= '2020-09-15'
  AND valid_to > '2020-09-15'

In [0]:
%sql
-- Comparar estado em 3 datas diferentes
SELECT '2020-08-01' AS data_consulta, status, sigla_orgao
FROM prata.tramitacoes_scd2
WHERE id_proposicao = 2258196 AND valid_from <= '2020-08-01' AND valid_to > '2020-08-01'
UNION ALL
SELECT '2021-01-01', status, sigla_orgao
FROM prata.tramitacoes_scd2
WHERE id_proposicao = 2258196 AND valid_from <= '2021-01-01' AND valid_to > '2021-01-01'
UNION ALL
SELECT '2023-06-01', status, sigla_orgao
FROM prata.tramitacoes_scd2
WHERE id_proposicao = 2258196 AND valid_from <= '2023-06-01' AND valid_to > '2023-06-01'

# Tempo Médio de Tramitação
Análise de duração por tipo de movimentação e comissão relatora.

In [0]:
%sql
CREATE OR REPLACE TABLE ouro.tempo_medio_tramitacao
USING DELTA
COMMENT 'Camada Ouro - Tempo medio de tramitacao por status e orgao'
AS
SELECT 
    status,
    sigla_orgao,
    COUNT(*) AS qtd_ocorrencias,
    ROUND(AVG(DATEDIFF(valid_to, valid_from)), 0) AS media_dias,
    MAX(DATEDIFF(valid_to, valid_from)) AS maximo_dias
FROM prata.tramitacoes_scd2
WHERE is_current = FALSE
  AND status IS NOT NULL
GROUP BY status, sigla_orgao
ORDER BY media_dias DESC

In [0]:
%sql
-- Top 10 tramitacoes mais demoradas
SELECT 
    status,
    sigla_orgao,
    qtd_ocorrencias,
    media_dias,
    maximo_dias
FROM ouro.tempo_medio_tramitacao
ORDER BY media_dias DESC
LIMIT 10

# Alertas Automáticos
Registrar quando uma proposição avança para Plenário ou é arquivada.

In [0]:
%sql
CREATE OR REPLACE TABLE ouro.alertas_tramitacao
USING DELTA
COMMENT 'Camada Ouro - Alertas quando PL avanca para Plenario ou e arquivada'
AS
SELECT 
    id_proposicao,
    valid_from AS data_evento,
    status,
    sigla_orgao,
    despacho,
    CASE 
        WHEN LOWER(status) LIKE '%plenário%' OR LOWER(sigla_orgao) = 'plen' THEN 'Avançou para Plenário'
        WHEN LOWER(status) LIKE '%arquiv%' THEN 'Arquivada'
        WHEN LOWER(status) LIKE '%transformad%' OR LOWER(status) LIKE '%lei%' THEN 'Transformada em Lei'
        ELSE 'Outro'
    END AS tipo_alerta
FROM prata.tramitacoes_scd2
WHERE LOWER(status) LIKE '%plenário%' 
   OR LOWER(sigla_orgao) = 'PLEN'
   OR LOWER(status) LIKE '%arquiv%'
   OR LOWER(status) LIKE '%transformad%'
   OR LOWER(status) LIKE '%lei%'
ORDER BY data_evento DESC

In [0]:
%sql
SELECT 
    id_proposicao,
    data_evento,
    status,
    sigla_orgao,
    tipo_alerta
FROM ouro.alertas_tramitacao
ORDER BY data_evento DESC
LIMIT 10